### Analysis

In [1]:
%cd ../

/nas/zhangtianning.di/projects/unique_data_build


In [2]:
from pathlib import Path

In [3]:
all_unarxive_json = Path('/nvme/zhangtianning/datasets/whole_arxiv_all_files/unprocessed_json/').glob("*/unarxive_clean/*.json")

In [4]:
all_unarxive_json = list(all_unarxive_json)

In [5]:
len(all_unarxive_json)

0

### Convert 

In [6]:
### update unarxive

def sort_section_numbers(section_numbers):
    def split_section_number(section_number):
        return list(map(int, section_number.split('.')))

    sorted_numbers = sorted(section_numbers, key=split_section_number)
    return sorted_numbers

In [7]:
section_numbers = ['2.2.1', '1.1', '2.2', '1', '1.2', '2', '1.1.1']
sorted_numbers = sort_section_numbers(section_numbers)
print(sorted_numbers)

['1', '1.1', '1.1.1', '1.2', '2', '2.2', '2.2.1']


In [ ]:
# done_file_names = [x.name for x in Path('/nvme/zhangtianning.di/datasets/unarxive/').glob('**/*.jsonl')]
# all_raw_files   = list(Path('/nvme/zhangtianning.di/datasets/unarxive/').glob('**/*.jsonl'))
# all_raw_files = [str(t) for t  in all_raw_files]
# with open("/nvme/zhangtianning.di/datasets/unarxive/whole_filelist.json",'w') as f:
#     json.dump(all_raw_files, f)

# done_file_names = [x.name for x in Path('/nvme/zhangtianning.di/datasets/unarxive/').glob('**/*.jsonl')]

In [11]:
all_raw_files   = list(Path('/nvme/zhangtianning.di/datasets/unarxive/').glob('**/*.jsonl'))

In [ ]:
from tqdm.auto import tqdm

#### convert unarxive format into uparxive format

from uparxive.xml_to_json.clean_unarXive_data import *

import tiktoken
encoding = tiktoken.get_encoding("cl100k_base")

In [17]:
##############
metadata_i    = data['metadata']
body_text_i   = data['body_text']
bib_entries_i = data['bib_entries']
ref_entries_i = data['ref_entries']
paper_id = data['paper_id']

NameError: name 'data' is not defined

In [16]:
# get a dict for format citation/reference data
insert_data = {}
type_list = []

# for formula, figure, table
figure_ref  = {}
table_ref   = {}
equation_ref= {}
figures_metadata ={}
tables_metadata  ={}

paper_unique_id  = identify_string_type(metadata_i['id'])
for _id, item in ref_entries_i.items():
    type_list.append(item['type'])
    key = "{}:{}".format(item['type'], _id)
    if item['type'] == 'formula':
        value = """${}$""".format(item['latex'])
    elif item['type'] == 'table':
        value = f"""[Table.{len(tables_metadata)} of {paper_unique_id}]"""
        tables_metadata[_id] = item
        table_ref[_id] =[ 'Table', len(tables_metadata)]
    elif item['type'] == 'figure':
        value = f"""[Figure.{len(figures_metadata)} of {paper_unique_id}]"""
        figures_metadata[_id] = item
        figure_ref[_id]  = ['Figure',len(figures_metadata)]
    else:
        print('='*20)
        print(item)
        print('='*20)
        value = ""
    insert_data[key] = value

# remove duplicate
type_list = list(set(type_list)) 

# for citation
type_list.append('cite')

for i,(_id, item) in enumerate(bib_entries_i.items()):
    key = "{}:{}".format('cite', _id)
    value = get_name(item)
    value = f"[Ref.{i} of {paper_unique_id}]" if value is None else value
    insert_data[key] = value

NameError: name 'metadata_i' is not defined

In [18]:
with open(all_raw_files[0]) as f:
    data = pd.read_json(path_or_buf=f, lines=True)

In [21]:
data.iloc[0]['bib_entries']

{'64222df7366a9d41ed0ca7394982da216b19672a': {'bib_entry_raw': 'J. S. Bell, Physics 1, 195 (1964); reprinted in W. Zurek and J. A. Wheeler, eds., Quantum Theory and Measurement, (Princeton University Press, Princeton, 1983).',
  'contained_arXiv_ids': [],
  'contained_links': [],
  'discipline': 'Physics',
  'ids': {'open_alex_id': 'https://openalex.org/W1987630316',
   'sem_open_alex_id': 'https://semopenalex.org/work/W1987630316',
   'pubmed_id': '',
   'pmc_id': '',
   'doi': '10.1119/1.13804',
   'arxiv_id': ''}},
 'cebf5fb6b85c2317258eb9214511e7dbc0bfeb16': {'bib_entry_raw': 'J. J. Sakurai, Modern Quantum Mechanics (Addison-Wesley, Reading Massachusetts, 1994).',
  'contained_arXiv_ids': [],
  'contained_links': [],
  'discipline': 'Physics',
  'ids': {'open_alex_id': 'https://openalex.org/W1500256440',
   'sem_open_alex_id': 'https://semopenalex.org/work/W1500256440',
   'pubmed_id': '',
   'pmc_id': '',
   'doi': '10.1017/9781108587280',
   'arxiv_id': ''}},
 '064cb4389cfd05e114

In [ ]:
all_text = []
Reference= []
ReferenceQ= False
acknowledgement = None
for paragraph_id, paragraph in enumerate(body_text_i):
    #print(paragraph.keys())
    text = paragraph['text']

    start_string = text.strip()
    if start_string.startswith('Acknowledgement'):
        acknowledgement = new_text
        continue
    if start_string.startswith('REFERENCES') or start_string.startswith('Reference'):
        ReferenceQ=True
        print(f"fail at paragraph {paragraph_id}")
        print(paragraph)
        with open('fail.json','w') as f:
            json.dump(data, f)
        raise

    new_text = format_text_with_values(text, insert_data, type_list,paper_unique_id)
    if len(new_text)==0: continue
    if not ReferenceQ:   
        paragraph['format_text'] = new_text
    else:
        Reference.append(new_text)

sections = []
structued_paragraph = {}
for flatten_paragraph in body_text_i:
    section_num  = flatten_paragraph['sec_number']
    section_num  = section_num.strip()
    if section_num not in structued_paragraph:
        structued_paragraph[section_num] = {'section_content':[]}

    section_name = flatten_paragraph['section']
    section_name = better_latex_sentense_string(section_name)

    if section_name in structued_paragraph[section_num]:
        assert section_name == structued_paragraph[section_num], f"why we get two different section name for [{section_name}] and ]"
    structued_paragraph[section_num]['section_name'] = section_name
    if 'format_text' not in flatten_paragraph:
        #print(flatten_paragraph)
        continue
    new_text = flatten_paragraph['format_text']
    all_text = structued_paragraph[section_num]['section_content']

    if all_text and new_text.strip() and (new_text.strip()[0].islower() or new_text.strip().startswith('Proof')):
        all_text[-1]+= ' ' + new_text.strip()
    elif all_text and all_text[-1].strip() and (all_text[-1].strip()[-1]=="$" or all_text[-1].strip()[-1]==":") and len(all_text[-1].split())<128:
        all_text[-1]+= ' ' + new_text
    else:
        all_text.append(new_text.replace('\n'," "))
    structued_paragraph[section_num]['section_content'] = all_text

In [26]:
data= data.iloc[0]

In [28]:
from uparxive.xml_to_json.clean_unarXive_data import identify_string_type, get_name,format_text_with_values
from uparxive.xml_to_json.xml_to_dense_text import better_latex_sentense_string
from uparxive.reference_reterive.Reference import UniqueID
from uparxive.batch_run_utils import BatchModeConfig, obtain_processed_filelist, process_files,dataclass,save_analysis
import json

In [29]:
root_dir = 'debug'
reterive_result_mode = False
paper_id='test'
metadata_i    = data['metadata']
body_text_i   = data['body_text']
bib_entries_i = data['bib_entries']
ref_entries_i = data['ref_entries']
paper_id = data['paper_id']
paper_id = paper_id.replace('/',"_")
output_dir = os.path.join(root_dir, paper_id, 'unarxive_clean')
Content_Path = os.path.join(output_dir, f'{paper_id}.retrieved.json') if reterive_result_mode else os.path.join(output_dir, f'{paper_id}.json')

#if os.path.exists(Content_Path):return
os.makedirs(os.path.dirname(Content_Path),exist_ok=True)
# get a dict for format citation/reference data
insert_data = {}
type_list = []

# for formula, figure, table
figure_ref  = {}
table_ref   = {}
equation_ref= {}
figures_metadata ={}
tables_metadata  ={}

paper_unique_id  = identify_string_type(metadata_i['id'])
for _id, item in ref_entries_i.items():
    type_list.append(item['type'])
    key = "{}:{}".format(item['type'], _id)
    if item['type'] == 'formula':
        value = """${}$""".format(item['latex'])
    elif item['type'] == 'table':
        value = f"""[Table.{len(tables_metadata)} of {paper_unique_id}]"""
        tables_metadata[_id] = item
        table_ref[_id] =[ 'Table', len(tables_metadata)]
    elif item['type'] == 'figure':
        value = f"""[Figure.{len(figures_metadata)} of {paper_unique_id}]"""
        figures_metadata[_id] = item
        figure_ref[_id]  = ['Figure',len(figures_metadata)]
    else:
        print('='*20)
        print(item)
        print('='*20)
        value = ""
    insert_data[key] = value

# remove duplicate
type_list = list(set(type_list)) 

# for citation
type_list.append('cite')

for i,(_id, item) in enumerate(bib_entries_i.items()):
    key = "{}:{}".format('cite', _id)
    value = get_name(item)
    value = f"[Ref.{i} of {paper_unique_id}]" #if value is None else value
    insert_data[key] = value

all_text = []
Reference= []
ReferenceQ= False
acknowledgement = None
for paragraph_id, paragraph in enumerate(body_text_i):
    #print(paragraph.keys())
    text = paragraph['text']

    start_string = text.strip()

    if start_string.lower().startswith('appendix') or start_string.lower().startswith('supplementary'):
        ReferenceQ=False
    if start_string.lower().startswith('reference'):
        ReferenceQ=True
        # print(f"fail at paragraph {paragraph_id}")
        # print(paragraph)
        # with open('fail.json','w') as f:
        #     json.dump(data, f)
        # raise

    new_text = format_text_with_values(text, insert_data, type_list,paper_unique_id)
    if start_string.startswith('Acknowledgement'):
        acknowledgement = new_text
        continue
    if len(new_text)==0: continue
    if not ReferenceQ:   
        paragraph['format_text'] = new_text
    else:
        Reference.append(new_text)

sections = []
structued_paragraph = {}
for flatten_paragraph in body_text_i:
    section_num  = flatten_paragraph['sec_number']
    section_num  = section_num.strip()
    if section_num not in structued_paragraph:
        structued_paragraph[section_num] = {'section_content':[]}

    section_name = flatten_paragraph['section']
    section_name = better_latex_sentense_string(section_name)
    if section_name and "{{" in section_name:
        section_name = format_text_with_values(section_name, insert_data, type_list,paper_unique_id)
    if section_name in structued_paragraph[section_num]:
        assert section_name == structued_paragraph[section_num], f"why we get two different section name for [{section_name}] and ]"
    structued_paragraph[section_num]['section_name'] = section_name
    if 'format_text' not in flatten_paragraph:
        #print(flatten_paragraph)
        continue
    new_text = flatten_paragraph['format_text']
    all_text = structued_paragraph[section_num]['section_content']

    if all_text and new_text.strip() and (new_text.strip()[0].islower() or new_text.strip().startswith('Proof')):
        all_text[-1]+= ' ' + new_text.strip()
    elif all_text and all_text[-1].strip() and (all_text[-1].strip()[-1]=="$" or all_text[-1].strip()[-1]==":") and len(all_text[-1].split())<128:
        all_text[-1]+= ' ' + new_text
    else:
        all_text.append(new_text.replace('\n'," "))
    structued_paragraph[section_num]['section_content'] = all_text


sections = []
appendix = []
section_num_keys = list(structued_paragraph.keys())

# print("\n"*3)
# print("=========>",section_num_keys)
# section_num_keys = sort_section_numbers(section_num_keys)


for section_num in section_num_keys:
    section_pool  = structued_paragraph[section_num]
    section_name   = section_pool['section_name']
    appendex_mode = False
    if (section_name and section_name.lower() in ['appendix','supplementary'] )or '-' in section_num:
        appendex_mode = True

    if not section_name: 
        if not appendex_mode:
            section_name = f'Section {section_num}'
        else:
            section_name = f"Appendix {section_num.replace('-','')}"
    now_section = {'section_title':section_name}|{'section_content':section_pool['section_content'],#split_by_indentation(section_pool['section_content']),
                                                  'section_num':section_num}
    if not appendex_mode:
        sections.append(now_section)
    else:
        appendix.append(now_section)

#cat_final_i, sec_final_i = concatenate_text(all_text, body_text_i, metadata_i, encoding=encoding)
#Reference = "|.|".join(Reference)

undo_citation_keys   = []
undo_citation_string = []
done_citation_keys   = []
done_citation_string = []
done_citation_doi    = []
for key, valpool in data['bib_entries'].items():

    citation = valpool.get('bib_entry_raw',"")
    if not citation:continue

    ids = valpool.get('ids',{})
    unique_id = UniqueID.from_dict(ids)
    if unique_id.is_nan():
        undo_citation_keys.append(key)  
        undo_citation_string.append(citation)  
    else:
        done_citation_keys.append(key)  
        done_citation_string.append(citation)
        done_citation_doi.append({k:val for k,val in unique_id.to_dict().items() if val})

bibitem_ref_metadata = {k:v for k,v in zip(undo_citation_keys, undo_citation_string)}
bibitem_ref_metadata = bibitem_ref_metadata| {k:v for k,v in zip(done_citation_keys, done_citation_string)}
whole_metadata = {'figures_metadata':figures_metadata,
                  'tables_metadata':tables_metadata,
                  'bibitem_ref_metadata':bibitem_ref_metadata}
whole_ref_to_labels = table_ref|figure_ref
abstract = better_latex_sentense_string(data.get('abstract')['text'])
output_dict = {'abstract':abstract,
               'acknowledge':acknowledgement,
               'sections':sections,
               'appendix':[],
               'metadata':whole_metadata,
               'paper_id':paper_id,
               'whole_ref_to_labels':whole_ref_to_labels,
               'missing_citation_labels':{}}

In [33]:
citation

'J. S. Bell, in B. d\'Espagnat, ed. Proc. Int. School of Physics "Enrico Fermi", (Academic Press: New York, 1971), p. 171.'

In [ ]:
import re
import roman  # You'll need to install the 'roman' package for this

# First, define a helper function to determine if a string is an integer
def is_integer(s):
    try:
        int(s)
        return True
    except ValueError:
        return False

# Define a helper function to determine if a string is a Roman numeral
def is_roman_numeral(s):
    try:
        roman.fromRoman(s)
        return True
    except roman.InvalidRomanNumeralError:
        return False

# Define a function to convert section parts to a sortable key
def section_key(section):
    # Split the section into parts (e.g., '2.3.1' -> ['2', '3', '1'])
    parts = re.split(r'\.', section)
    key = []
    
    for part in parts:
        if is_integer(part):  # Integer case
            key.append(('int', int(part)))
        elif is_roman_numeral(part):  # Roman numeral case
            key.append(('roman', roman.fromRoman(part)))
        elif part.isalpha():  # Letter case
            key.append(('letter', part.upper()))  # treat 'a' and 'A' equally
        else:
            raise ValueError(f"Unknown section format: {part}")
    
    return key

# Now you can sort a list of section numbers
def sort_sections(sections):
    return sorted(sections, key=section_key)

# Example```python
import re

# Helper functions to identify and convert section numbering formats
def int_or_roman_to_int(value):
    # Converts integers and roman numerals to integers for comparison
    try:
        return int(value)
    except ValueError:
        # Assume roman numeral since it's not an integer
        return roman_to_int(value)

def roman_to_int(value):
    # Converts roman numerals to integers
    roman_numerals = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    int_val = 0
    for i in range(len(value)):
        if i > 0 and roman_numerals[value[i]] > roman_numerals[value[i - 1]]:
            int_val += roman_numerals[value[i]] - 2 * roman_numerals[value[i - 1]]
        else:
            int_val += roman_numerals[value[i]]
    return int_val

def section_key(section):
    # Converts a section into a tuple of keys for sorting
    parts = re.split('\.|-', section)  # Split on . or -
    key_parts = []
    
    for part in parts:
        if part.isdigit():  # Integer part
            key_parts.append(int(part))
        elif part.isalpha():  # Letter part
            # Convert letters to integers based on alphabetical position
            # Make sure to handle lower and uppercase letters identically
            key_parts.append(ord(part.upper()) - ord('A') + 1)
        elif re.match(r'^[IVXLCDM]+$', part):  # Roman numeral part
            key_parts.append(roman_to_int(part))
        else:
            # Handle invalid section formats
            raise ValueError(f"Invalid section format: {part}")
    
    return tuple(key_parts)

# Sort a given list of section numbers
def sort_section_numbers(section_numbers):
    return sorted(section_numbers, key=section_key)

# Example usage
section_numbers = ["1", "2", "3.1", "3.2", "10", "A", "B", "C", "I", "II", "III", "X", "V"]
sorted_sections = sort_section_numbers(section_numbers)
print(sorted_sections)

In [ ]:
import re

def sort_section_numbers(section_numbers):
    def split_section_number(section_number):
        parts = re.split(r'(\d+)', section_number)
        return [int(part) if part.isdigit() else part for part in parts]

    sorted_numbers = sorted(section_numbers, key=split_section_number)
    return sorted_numbers

section_numbers = ['2.2.1', '1.1', '2.2', '1', '1.2', '2', '1.1.1', '2.2.A', '2.B', '1.1.C']
sorted_numbers = sort_section_numbers(section_numbers)
print(sorted_numbers)

In [ ]:
sections = []
appendix = []
section_num_keys = list(structued_paragraph.keys())
section_num_keys = sort_section_numbers(section_num_keys)

for section_num in section_num_keys:
    section_pool  = structued_paragraph[section_num]
    section_name   = section_pool['section_name']
    appendex_mode = False
    if section_name.lower() in ['appendix','supplementary'] or '-' in section_num:
        appendex_mode = True

    if not section_name: 
        if not appendex_mode:
            section_name = f'Section {section_num}'
        else:
            section_name = f"Appendix {section_num.replace('-','')}"
    now_section = {'section_title':section_name}|{'section_content':split_by_indentation(section_pool['section_content']),
                                                  'section_num':section_num}
    if not appendex_mode:
        sections.append(now_section)
    else:
        appendix.append(now_section)

In [ ]:
from python_script.reference_reterive.Reference import flatten_dict
from xml_to_dense_text import better_latex_sentense_string
class BaseElement:
    def to_dict(self):
        return vars(self)
    
    def to_flatten_dict(self):
        out = flatten_dict(self.to_dict())
        
        return out
    

@dataclass
class Section(BaseElement):
    section_title: str = None
    section_content: List[str] = None

@dataclass
class PaperMetadata(BaseElement):
    figures_metadata: dict = None
    tables_metadata: dict = None
    bibitem_ref_metadata: dict = None

@dataclass
class Paper(BaseElement):
    paper_id: str = None
    abstract: str = None
    acknowledge: str = None
    sections: List[Section] = None
    appendix: List[Section] = None
    metadata: PaperMetadata = None
    whole_ref_to_labels: Dict[str, list] = None
    missing_citation_labels: dict = None
    def to_dict(self):
        return vars(self)
    
    def to_flatten_dict(self):
        out = flatten_dict(self.to_dict())
        return out
    @staticmethod
    def get_paper_id(pool):
        return pool.get('paper_id', None)
    
    
    
    @staticmethod
    def abstract(pool):
        abstract =  pool.get('abstract', None)
        if isinstance(abstract, dict):
            abstract = abstract['text']
        abstract = better_latex_sentense_string(abstract)
        return abstract


#os.makedirs(output_dir, exist_ok=True)

#Content_Paht = os.path.join(output_dir, f'{_paper_id}.retrieved.json') if reterive_result_mode else os.path.join(output_dir, f'{_paper_id}.json')
# with open(Content_Paht, 'w') as f:json.dump(output_dict, f, indent=2)
# if not reterive_result_mode:
#     keys  = list(bibitem_ref_metadata.keys())
#     citation_string = [bibitem_ref_metadata[key] for key in keys]
#     with open(os.path.join(output_dir, f'reference.keys'), 'w') as f:
#         for key in keys:f.write(key+'\n')
#     with open(os.path.join(output_dir, f'reference.txt'), 'w') as f:
#         for string in citation_string:f.write(string+'\n')
#     with open(os.path.join(output_dir, f'bibitem_ref_metadata_not_in_context.json'), 'w') as f:
#         json.dump(bibitem_ref_metadata_not_in_context, f, indent=2)
#     with open(os.path.join(output_dir, f'note_ref_metadata_not_in_context.json'), 'w') as f:
#         json.dump(note_ref_metadata_not_in_context, f, indent=2)